In [1]:
# Install compatible dependencies
!pip install --upgrade numpy pandas scikit-learn peft wandb tqdm -q

In [ ]:
import os
os.kill(os.getpid(), 9)

# SOMA — Experiment 1: Permuted MNIST

**Wakasa Labs · Nairobi, Kenya · March 2026**

This notebook runs the full SOMA system on 10 permuted MNIST tasks.
Designed to run on **Kaggle T4 GPU** (~15 min runtime).

**PASS criterion:** BT > -0.05 AND K < 10

In [2]:
# Clone SOMA repo if not exists and add to path
import os
import sys

if not os.path.exists('soma_research'):
    !git clone https://github.com/LensenWakasa/SOMA-research.git soma_research

sys.path.insert(0, os.path.abspath('soma_research'))

# Verify import
from soma.core.necessity import SomaNecessity
from soma.core.grow import SomaGrow
from soma.core.learn import SomaLearn
print('SOMA loaded successfully')

SOMA loaded successfully


In [5]:
# GPU check
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Fixed attribute name from total_mem to total_memory
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# Pull latest SOMA code with fixes (MERGE masking, warmup guard, BT tracking)
import subprocess
import sys
import os
import importlib

os.chdir('soma_research')
subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True)
os.chdir('..')

# Reload all SOMA modules to pick up latest code
modules_to_reload = [
    'soma.core.necessity',
    'soma.core.grow', 
    'soma.core.learn',
    'soma.core.router',
    'soma.experiments.run_permuted_mnist'
]
for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

# Run experiment
from soma.experiments.run_permuted_mnist import run_experiment
import argparse
import torch

args = argparse.Namespace(
    n_tasks=10, n_train=1000, n_test=200,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42, no_rl=False, disable_n1=False, disable_n2=False, disable_n3=False,
)

result = run_experiment(args)

In [21]:
# Verify PASS/FAIL
bt = result['backward_transfer']
k = result['final_k']
passed = bt > -0.05 and k < 10
print(f'\nResult: {"PASS" if passed else "FAIL"}')
print(f'BT = {bt:.4f} (target > -0.05)')
print(f'K  = {k} (target < 10)')


Result: FAIL
BT = -0.2328 (target > -0.05)
K  = 3 (target < 10)
